<h2>Build an AI Math Assistant with LangChain Tool Calling</h2>

In this lab, you will learn how to build a simple agent with LangChain, enabling AI agents to perform specific tasks. You'll create a mathematical toolkit that allows an AI agent to perform basic arithmetic operations through natural language interaction.

Through this lab, you'll build an agent that can understand and solve mathematical queries like "add 25 and 15, then multiply by 2" by breaking down complex operations into simple steps.



In [ ]:
%pip install langchain==0.3.23
%pip install langchain-ibm==0.3.10
%pip install langchain-community==0.3.16
%pip install wikipedia==1.4.0
%pip install openai==1.77.0
%pip install langchain-openai==0.3.16

In [34]:
from langchain.agents import AgentType
import re

def add_numbers(inputs:str) -> dict:
    """
    Adds a list of numbers provided in the input dictionary or extracts numbers from a string.

    Parameters:
    - inputs (str): 
    string, it should contain numbers that can be extracted and summed.

    Returns:
    - dict: A dictionary with a single key "result" containing the sum of the numbers.

    Example Input (Dictionary):
    {"numbers": [10, 20, 30]}

    Example Input (String):
    "Add the numbers 10, 20, and 30."

    Example Output:
    {"result": 60}
    """
    numbers = [int(x) for x in inputs.replace(",", "").split() if x.isdigit()]

    result = sum(numbers)
    return {"result": result}

def add_numbers_basic(inputs:str) -> int:
    """
    Adds a list of numbers provided in the input dictionary or extracts numbers from a string.

    Parameters:
    - inputs (str): 
    string, it should contain numbers that can be extracted and summed.

    Returns:
    - int: The sum of the numbers.

    
    Example Input (Dictionary):
    {"numbers": [10, 20, 30]}

    Example Input (String):
    "Add the numbers 10, 20, and 30."

    Example Output:
    60

    
    CRITICAL USAGE RULES:
    1. Do not call this tool a second time to "verify" or "double-check" or "confirm" the result or observation.
    2. Once you receive the output int, immediately proceed to your next step without any further calls to this tool.
    
    """
    numbers = [int(x) for x in inputs.replace(",", "").split() if x.isdigit()]

    result = sum(numbers)
    return result

    

add_numbers("1 2") 

{'result': 3}

Loading the LLM: Choosing the right language model
In this example, IBM’s ChatWatsonxwill be used to load a language model (LLM) for interacting with tools. IBM’s models, like Granite 4, are highly versatile and excel at advanced reasoning tasks.

That said, other providers offer LLMs with different strengths:

OpenAI (GPT-4/GPT-3.5): Best for versatility and advanced reasoning.
Facebook (Meta, LLaMA): Open-access, highly customizable for specialized use cases.
IBM watsonx Granite: Ideal for enterprise applications with seamless integration.
Anthropic (Claude): Focused on safety, reliability, and ethical AI.
Cohere: Affordable and efficient for lightweight, task-specific models.

In [27]:
from langchain.agents import Tool
add_tool=Tool(
        name="AddTool",
        func=add_numbers,
        description="Adds a list of numbers and returns the result.")

print("tool object",add_tool)

tools = [
    Tool(
        name="Calculator",
        func=add_tool,
        description="Useful for math calculations",
    )
]

tools_basic = [
    Tool(
        name="Calculator",
        func=add_numbers_basic,
        description="Useful for math calculations",
    )
]

tool object name='AddTool' description='Adds a list of numbers and returns the result.' func=<function add_numbers at 0x00000265640F09D0>


In [4]:
# Tool name
print("Tool Name:")
print(add_tool.name)

# Tool description
print("Tool Description:")
print(add_tool.description)

# Tool function
print("Tool Function:")
print(add_tool.invoke)


Tool Name:
AddTool
Tool Description:
Adds a list of numbers and returns the result.
Tool Function:
<bound method BaseTool.invoke of Tool(name='AddTool', description='Adds a list of numbers and returns the result.', func=<function add_numbers at 0x000002655FD8DBD0>)>


In [5]:
print("Calling Tool Function:")
test_input = "10 20 30 a b" 
print(add_tool.invoke(test_input))  # Example

Calling Tool Function:
{'result': 60}


In [ ]:
%pip install langchain_ollama==0.3.3

In [35]:
from langchain.agents import AgentType, initialize_agent
from langchain.tools import Tool
from langchain_ollama import ChatOllama  # or langchain_community.chat_models


# 1. Initialize the local Llama model (make sure Ollama is running with the model pulled)
llm = ChatOllama(model="llama3", temperature=0)



# 3. Pass the Llama llm instance into initialize_agent
agent = initialize_agent(
    tools=tools_basic, # tools fails since current agent executor of lang_chain is not able to accept output in dict format (have to use latest langchain version modern agent executor)
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=3,           # Raise from default to allow more steps
    max_execution_time=60        # Raise to allow up to 60 seconds of processing

)

'''
response = agent.invoke({
    "input" : "Add the numbers 10, 20, and 30" # WORKS
})
print(response) # WORKS
'''

response = agent.invoke({
    "input":"What is 25 + 18?" # FAILS
})
print(response)




> Entering new AgentExecutor chain...
Let's get started!

Question: What is 25 + 18?
Thought: Hmm, this looks like a simple math problem. I think I should use my Calculator tool to solve it.

Action: Calculator
Action Input: "25 + 18"
Observation: 43
Thought:Thought: Ah, the Calculator tool gave me the result! Now I just need to confirm that this is indeed the correct answer.

Action: None (just confirming the observation)
Action Input: N/A
Observation: None (just confirming the observation) is not a valid tool, try one of [Calculator].
Thought:Let's continue!

Question: What is 25 + 18?
Thought: Hmm, this looks like a simple math problem. I think I should use my Calculator tool to solve it.

Action: Calculator
Action Input: "25 + 18"
Observation: 43
Thought:

> Finished chain.
{'input': 'What is 25 + 18?', 'output': 'Agent stopped due to iteration limit or time limit.'}
